In [ ]:
import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
device = "cuda:1" if torch.cuda.is_available() else "cpu"
# bfloat16 is supported on Ampere GPUs (Compute Capability 8.0+) 
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

# Initialize the model and load the pretrained weights.
# This will automatically download the model weights the first time it's run, which may take a while.
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)

In [ ]:
import os
import re

image_names = os.listdir("./extracted_frames")
image_names = [os.path.join("./extracted_frames", name) for name in image_names if name.endswith('.jpg') or name.endswith('.png')]

# 按照时间戳排序
def extract_timestamp(filename):
    # 提取文件名中的时间戳部分，格式如: frame_000_40753679_6789.515.png
    basename = os.path.basename(filename)
    # 使用正则表达式提取时间戳，确保不包含文件扩展名前的点
    match = re.search(r'frame_\d+_\d+_(\d+\.\d+)', basename)
    if match:
        return float(match.group(1))
    else:
        # 如果没有匹配到时间戳，使用文件名排序
        return float('inf')

image_names.sort(key=extract_timestamp)
print(f"按时间戳排序后的前5个文件: {[os.path.basename(name) for name in image_names[:5]]}")

# Load and preprocess the images.
images = load_and_preprocess_images(image_names).to(device)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, List, Tuple
import cv2
from matplotlib.patches import Rectangle

def create_attention_visualizer(model, target_layers=[0, 5, 11, 17, 23]):
    """
    创建一个优雅的attention map可视化器，只选择特定层以节省内存
    """
    attention_maps = {}
    hooks = []
    
    def create_hook(name):
        def hook(module, input, output):
            # 捕获attention权重
            if hasattr(module, 'attn') and hasattr(module.attn, 'qkv'):
                x = input[0]
                B, N, C = x.shape
                
                # 为了节省内存，只处理较小的batch
                if B > 1:
                    x = x[:1]  # 只取第一个batch
                    B = 1
                
                qkv = module.attn.qkv(x).reshape(B, N, 3, module.attn.num_heads, module.attn.head_dim).permute(2, 0, 3, 1, 4)
                q, k, v = qkv.unbind(0)
                
                # 只选择部分头以节省内存
                num_heads_to_keep = min(8, module.attn.num_heads)
                q = q[:, :num_heads_to_keep]
                k = k[:, :num_heads_to_keep]
                
                # 计算attention权重
                attn = (q @ k.transpose(-2, -1)) * module.attn.scale
                attn = attn.softmax(dim=-1)
                
                # 立即移到CPU并删除GPU张量
                attention_maps[name] = {
                    'attention': attn.detach().cpu(),
                    'tokens_shape': (B, N, C),
                    'num_heads': num_heads_to_keep
                }
                
                # 清理GPU内存
                del attn, q, k, v, qkv
                torch.cuda.empty_cache()
                
        return hook
    
    # 只在指定的层上注册hooks
    for i in target_layers:
        if i < len(model.aggregator.frame_blocks):
            hook_name = f'frame_block_{i}'
            hook = model.aggregator.frame_blocks[i].register_forward_hook(create_hook(hook_name))
            hooks.append(hook)
        
        if i < len(model.aggregator.global_blocks):
            hook_name = f'global_block_{i}'
            hook = model.aggregator.global_blocks[i].register_forward_hook(create_hook(hook_name))
            hooks.append(hook)
    
    return attention_maps, hooks



In [ ]:
attention_maps, hooks = create_attention_visualizer(model, target_layers=[0,3, 5, 9,11,15, 17,21, 23])

print("Attention可视化器已创建，现在重新运行模型以捕获attention maps...")
print("只监控层: [0, 3, 5, 9, 11, 15, 17, 21, 23] 以节省内存")
with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=dtype):
        # 清空之前的attention maps
        attention_maps.clear()
        # 为了节省内存，只处理前几张图像
        images_subset = images[:6] if images.shape[0] > 6 else images
        # 重新预测以捕获attention
        predictions_with_attn = model(images_subset)
        
        # 立即清理GPU内存
        torch.cuda.empty_cache()

print(f"捕获到 {len(attention_maps)} 个attention maps")
print("可用的attention maps:", list(attention_maps.keys()))


In [ ]:
def analyze_attention_statistics(attention_maps: Dict):
    """
    分析attention maps的统计信息
    """
    print("=== VGGT Attention Maps 统计分析 ===")
    print(f"总共捕获的attention maps: {len(attention_maps)}")
    
    frame_layers = []
    global_layers = []
    
    for key in attention_maps.keys():
        if 'frame_block_' in key:
            layer_idx = int(key.split('_')[-1])
            frame_layers.append(layer_idx)
        elif 'global_block_' in key:
            layer_idx = int(key.split('_')[-1])
            global_layers.append(layer_idx)
    
    print(f"Frame attention layers: {sorted(frame_layers)}")
    print(f"Global attention layers: {sorted(global_layers)}")
    
    # 分析attention权重的分布
    for key, data in attention_maps.items():
        attention = data['attention']  # [B, num_heads, N, N]
        print(f"\n{key}:")
        print(f"  Shape: {attention.shape}")
        print(f"  Number of heads: {data['num_heads']}")
        print(f"  Attention weight range: [{attention.min():.4f}, {attention.max():.4f}]")
        print(f"  Mean attention: {attention.mean():.4f}")
        print(f"  Std attention: {attention.std():.4f}")
        
        # 分析attention的集中程度 (entropy)
        attn_entropy = -(attention * torch.log(attention + 1e-8)).sum(dim=-1).mean()
        print(f"  Average attention entropy: {attn_entropy:.4f}")

# 运行统计分析
# analyze_attention_statistics(attention_maps)

In [ ]:
# 导入自定义的attention工具函数
from attn_util import extract_pose_token_attention, visualize_pose_attention, analyze_pose_attention_stats

In [ ]:
# 提取pose token对每个图像的attention maps
pose_attention_maps = extract_pose_token_attention(attention_maps, images_subset)

# 显示统计信息
# analyze_pose_attention_stats(pose_attention_maps)

In [ ]:
# 重新加载更新后的attn_util模块
import importlib
import attn_util
importlib.reload(attn_util)
from attn_util import (extract_pose_token_attention, visualize_pose_attention, 
                      analyze_pose_attention_stats, visualize_cross_image_attention,
                      visualize_complete_global_attention)

# 重新提取pose token attention maps（现在正确处理每个图像的pose token）
pose_attention_maps = extract_pose_token_attention(attention_maps, images_subset)

print("提取的attention类型:")
print(f"Frame attention layers: {list(pose_attention_maps['frame_attention'].keys())}")
print(f"Global attention layers: {list(pose_attention_maps['global_attention'].keys())}")

# 显示详细统计信息
analyze_pose_attention_stats(pose_attention_maps)

In [ ]:
# 可视化每个图像的pose token对图像的attention maps
# 现在会分别显示frame attention（图像内部）和global attention（跨图像）

print("=== 1. 显示 Frame Attention （图像内部注意力） ===")
visualize_pose_attention(pose_attention_maps, images_subset, max_images=6, attention_type='frame')

print("\n=== 2. 显示 Global Attention （跨图像注意力 - 只显示自注意力） ===")
visualize_pose_attention(pose_attention_maps, images_subset, max_images=6, attention_type='global')

In [ ]:
# 单独查看某些层的attention模式
# 现在会分别显示frame和global attention
# selected_layers = ['frame_block_0', 'frame_block_11', 'frame_block_23', 'global_block_0', 'global_block_11', 'global_block_23']
# visualize_pose_attention(pose_attention_maps, images_subset, 
#                         layer_names=selected_layers, max_images=6)

print("\n=== 3. 完整的Global Attention可视化 （每个pose token对所有图像的attention） ===")
print("警告：这将生成大量图像！对于6个图像和5层，将有180个子图")

# 只显示部分层以减少输出
selected_global_layers = ['global_block_11', 'global_block_23']  # 只选择两层
visualize_complete_global_attention(pose_attention_maps, images_subset, 
                                  layer_names=selected_global_layers, max_images=4)  # 只显示4个图像

# 如果你想看所有层和所有图像，可以取消注释下面的代码：
# visualize_complete_global_attention(pose_attention_maps, images_subset, max_images=6)

In [ ]:

# 也可以单独查看某个图像的pose token对所有其他图像的attention
print("\n=== 4. 单独查看特定图像的Cross-Image Attention ===")

# 查看第0个图像的pose token对所有图像的attention
print("图像0的pose token对所有图像的attention:")
visualize_cross_image_attention(pose_attention_maps, images_subset, 
                               source_img_idx=0, layer_name='global_block_23')

# 查看第1个图像的pose token
if images_subset.shape[0] > 1:
    print("\n图像1的pose token对所有图像的attention:")
    visualize_cross_image_attention(pose_attention_maps, images_subset, 
                                   source_img_idx=1, layer_name='global_block_23')

In [ ]:
# =============================================================================
# 🔍 深度分析：Patch-to-Patch跨图像对应关系 
# =============================================================================

print("🚀 开始深度分析VGGT的多视图几何建模机制...")
print("=" * 60)

# 重新导入所有分析函数
import importlib
import attn_util
importlib.reload(attn_util)

# 导入patch-to-patch分析函数
try:
    from attn_util import (
        analyze_cross_patch_correspondence,
        visualize_patch_correspondence_matrix,
        analyze_geometric_correspondence_quality,
        analyze_correspondence_evolution
    )
    print("✅ 成功导入所有分析函数")
except ImportError as e:
    print(f"❌ 导入失败: {e}")
    print("请确保attn_util.py中包含了所有必要的分析函数")

In [ ]:
# 1. 跨图像Patch对应关系分析
print("\n" + "=" * 60)
print("🔍 分析跨图像Patch对应关系 - 这是理解多视图几何的核心！")
print("=" * 60)

# 分析最后一层的global attention中的patch对应关系
try:
    # 直接从原始attention maps分析patch对应关系
    S, C, H, W = images_subset.shape
    patch_size = 14
    patch_h, patch_w = H // patch_size, W // patch_size
    
    for layer_name, data in attention_maps.items():
        if 'global_block_23' not in layer_name:
            continue
            
        print(f"\n📊 分析 {layer_name} 的跨图像patch对应关系:")
        
        attention = data['attention']  # [B, num_heads, N, N]
        if attention.dim() == 4:
            attention = attention[0].mean(dim=0)  # 平均所有heads
        
        N = attention.shape[0]
        tokens_per_frame = 1 + 4 + patch_h * patch_w  # camera + register + patches
        
        if N != S * tokens_per_frame:
            print(f"⚠️ Token数量不匹配: 期望{S * tokens_per_frame}, 实际{N}")
            continue
        
        print(f"✅ Token结构正确: {S}个图像 × {tokens_per_frame}个tokens/图像")
        
        # 分析每对图像之间的patch对应关系
        correspondence_matrix = torch.zeros(S, S)
        
        for source_frame in range(S):
            for target_frame in range(S):
                source_patch_start = source_frame * tokens_per_frame + 5
                source_patch_end = source_patch_start + patch_h * patch_w
                target_patch_start = target_frame * tokens_per_frame + 5
                target_patch_end = target_patch_start + patch_h * patch_w
                
                # 提取patch-to-patch attention
                patch_to_patch = attention[source_patch_start:source_patch_end, 
                                         target_patch_start:target_patch_end]
                
                if patch_to_patch.shape == (patch_h * patch_w, patch_h * patch_w):
                    # 计算对应强度：每个source patch的最大attention值的平均
                    max_correspondences = patch_to_patch.max(dim=1)[0]
                    avg_correspondence_strength = max_correspondences.mean().item()
                    correspondence_matrix[source_frame, target_frame] = avg_correspondence_strength
                    
                    print(f"  Frame{source_frame}→Frame{target_frame}: "
                          f"平均对应强度 = {avg_correspondence_strength:.4f}")
                    
                    # 找到最强的几个对应关系
                    top_k = 5
                    top_values, top_indices = torch.topk(patch_to_patch.flatten(), k=top_k)
                    print(f"    Top {top_k} 最强对应: {top_values.tolist()}")
        
        # 可视化对应矩阵
        plt.figure(figsize=(8, 6))
        plt.imshow(correspondence_matrix.numpy(), cmap='hot', interpolation='nearest')
        plt.colorbar(label='correspondence strength')
        plt.title(f'{layer_name}: cross image patch correspondence strength matrix') # 英文：cross image patch correspondence strength matrix
        plt.xlabel('target image')
        plt.ylabel('origin image')
        
        # 添加数值标注
        for i in range(S):
            for j in range(S):
                plt.text(j, i, f'{correspondence_matrix[i,j]:.3f}', 
                        ha='center', va='center', color='white' if correspondence_matrix[i,j] > 0.5 else 'black')
        
        plt.tight_layout()
        plt.show()
        
        break  # 只分析一层

except Exception as e:
    print(f"❌ 分析过程中出错: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# 2. 几何对应关系质量分析
print("\n" + "=" * 60)
print("📐 几何对应关系质量定量分析")
print("=" * 60)

try:
    for layer_name, data in attention_maps.items():
        if 'global_block' not in layer_name:
            continue
            
        print(f"\n🔍 {layer_name} 几何一致性分析:")
        
        attention = data['attention']
        if attention.dim() == 4:
            attention = attention[0].mean(dim=0)
        
        N = attention.shape[0]
        tokens_per_frame = 1 + 4 + patch_h * patch_w
        
        if N != S * tokens_per_frame:
            continue
        
        # 分析Camera token的注意力分布模式
        print("  📊 Camera Token注意力分布分析:")
        
        # 1. 自注意力 vs 跨图像注意力比例
        self_attention_ratios = []
        cross_attention_ratios = []
        
        for source_idx in range(S):
            camera_token_idx = source_idx * tokens_per_frame  # Camera token位置
            camera_attention = attention[camera_token_idx]  # 该camera token的所有attention
            
            # 计算对自己图像patches的attention
            self_patch_start = source_idx * tokens_per_frame + 5
            self_patch_end = self_patch_start + patch_h * patch_w
            self_attn = camera_attention[self_patch_start:self_patch_end].sum().item()
            
            # 计算对其他图像patches的attention
            cross_attn = 0
            for target_idx in range(S):
                if target_idx != source_idx:
                    target_patch_start = target_idx * tokens_per_frame + 5
                    target_patch_end = target_patch_start + patch_h * patch_w
                    cross_attn += camera_attention[target_patch_start:target_patch_end].sum().item()
            
            total_patch_attn = self_attn + cross_attn
            if total_patch_attn > 0:
                self_ratio = self_attn / total_patch_attn
                cross_ratio = cross_attn / total_patch_attn
                
                self_attention_ratios.append(self_ratio)
                cross_attention_ratios.append(cross_ratio)
                
                print(f"    Camera{source_idx}: 自注意力={self_ratio:.3f}, 跨图像注意力={cross_ratio:.3f}")
        
        # 2. 总结统计
        if self_attention_ratios:
            avg_self_ratio = np.mean(self_attention_ratios)
            avg_cross_ratio = np.mean(cross_attention_ratios)
            
            print(f"  📈 层级总结:")
            print(f"    平均自注意力比例: {avg_self_ratio:.3f}")
            print(f"    平均跨图像注意力比例: {avg_cross_ratio:.3f}")
            
            # 3. 几何一致性评估
            print(f"  🎯 几何一致性评估:")
            if avg_self_ratio > 0.4:
                print(f"    ✓ 高自注意力比例 ({avg_self_ratio:.3f}) -> 强局部几何理解")
            else:
                print(f"    ⚠ 低自注意力比例 ({avg_self_ratio:.3f}) -> 局部几何理解较弱")
                
            if avg_cross_ratio > 0.2:
                print(f"    ✓ 充分跨图像注意力 ({avg_cross_ratio:.3f}) -> 良好的视图间对应")
            else:
                print(f"    ⚠ 跨图像注意力不足 ({avg_cross_ratio:.3f}) -> 视图间对应较弱")
            
            # 计算平衡度
            balance_score = 1 - abs(avg_self_ratio - avg_cross_ratio)
            print(f"    🎖 注意力平衡度: {balance_score:.3f} (越接近1越好)")

except Exception as e:
    print(f"❌ 几何质量分析出错: {e}")

In [ ]:
# 3. Attention演化过程分析
print("\n" + "=" * 60)
print("📈 Attention演化过程：从局部到全局的学习过程")
print("=" * 60)

try:
    # 选择早期、中期、晚期层进行对比
    global_layers = [k for k in attention_maps.keys() if 'global_block' in k]
    global_layers.sort(key=lambda x: int(x.split('_')[-1]))
    
    if len(global_layers) >= 3:
        early_layer = global_layers[0]
        mid_layer = global_layers[len(global_layers)//2]
        late_layer = global_layers[-1]
        
        selected_layers = [early_layer, mid_layer, late_layer]
        layer_names = ["早期层", "中期层", "晚期层"]
        
        print(f"选择的分析层: {selected_layers}")
        
        # 分析每层的attention集中程度
        for layer_key, display_name in zip(selected_layers, layer_names):
            data = attention_maps[layer_key]
            attention = data['attention']
            if attention.dim() == 4:
                attention = attention[0].mean(dim=0)
            
            N = attention.shape[0]
            tokens_per_frame = 1 + 4 + patch_h * patch_w
            
            if N != S * tokens_per_frame:
                continue
            
            print(f"\n🔍 {display_name} ({layer_key}):")
            
            # 计算Camera token attention的熵（衡量集中程度）
            camera_entropies = []
            
            for frame_idx in range(S):
                camera_token_idx = frame_idx * tokens_per_frame
                camera_attention = attention[camera_token_idx]
                
                # 只看对patch tokens的attention
                all_patch_attention = []
                for target_frame in range(S):
                    patch_start = target_frame * tokens_per_frame + 5
                    patch_end = patch_start + patch_h * patch_w
                    all_patch_attention.append(camera_attention[patch_start:patch_end])
                
                combined_attention = torch.cat(all_patch_attention)
                # 计算熵
                entropy = -(combined_attention * torch.log(combined_attention + 1e-8)).sum().item()
                camera_entropies.append(entropy)
            
            avg_entropy = np.mean(camera_entropies)
            print(f"    Camera Token平均熵: {avg_entropy:.3f}")
            print(f"    → 熵值解释: 越低表示attention越集中，几何理解越精确")
            
            # 计算注意力的空间分布方差
            spatial_variances = []
            for frame_idx in range(S):
                camera_token_idx = frame_idx * tokens_per_frame
                self_patch_start = frame_idx * tokens_per_frame + 5
                self_patch_end = self_patch_start + patch_h * patch_w
                self_attention = attention[camera_token_idx, self_patch_start:self_patch_end]
                
                # 重塑为2D attention map
                attention_2d = self_attention.reshape(patch_h, patch_w)
                spatial_var = torch.var(attention_2d).item()
                spatial_variances.append(spatial_var)
            
            avg_spatial_var = np.mean(spatial_variances)
            print(f"    空间注意力方差: {avg_spatial_var:.6f}")
            print(f"    → 方差值解释: 越高表示注意力分布越不均匀，越有选择性")
    
    else:
        print("⚠️ 需要至少3个global层来分析演化过程")

except Exception as e:
    print(f"❌ 演化分析出错: {e}")

# 4. 总结与洞察
print("\n" + "=" * 60)
print("🎯 深度分析总结：VGGT多视图几何建模机制")
print("=" * 60)

print("""
🔍 关键发现与洞察:

1. **Token结构设计**:
   - VGGT使用共享的Camera token设计，而非每图像独立pose token
   - 这种设计体现了相机pose不是独立估计，而是全局整合的结果

2. **Patch-to-Patch对应关系**:
   - 高对应强度区域揭示了模型找到的跨视图3D点对应
   - 对应位置分布展示了几何变换关系的学习
   - 对应强度矩阵反映了模型对场景结构的理解

3. **几何一致性机制**:
   - 自注意力比例体现了局部几何理解能力
   - 跨图像注意力展示了视图间对应建立能力
   - 注意力平衡度反映了局部-全局信息整合质量

4. **层次化学习过程**:
   - 早期层：主要进行局部特征提取
   - 中期层：开始建立跨视图对应关系
   - 晚期层：形成全局几何约束和一致性

🎖 **VGGT的多视图几何建模核心机制**:
- 通过Global Attention建立跨视图特征对应
- 利用Camera Token整合全局几何信息
- 实现从局部特征到全局几何约束的层次化学习

💡 **实际应用启示**:
- 对应强度高的区域是可靠的3D重建点
- 注意力演化过程反映了几何理解的渐进式建立
- 平衡的注意力分布预示着更好的重建质量
""")

print("🎉 分析完成！现在你应该清楚地理解了VGGT如何在多视图几何中建模局部-全局关系！")